In [12]:
!pip install langchain==0.2.16 langchain-community==0.2.16 langchain-huggingface langchain-chroma langchain-groq langchain-text-splitters chromadb sentence-transformers groq gradio pypdf -q

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain.chains import RetrievalQA
from langchain_groq import ChatGroq
import gradio as gr

print("All imports successful ✓")

All imports successful ✓


In [2]:
import os
os.environ["GROQ_API_KEY"] = "your_actual_key_here"

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)
response = llm.invoke("Say hello in one sentence")
print(response.content)

Hello, how can I assist you today?


In [3]:
import os

os.makedirs("documents", exist_ok=True)

doc1 = """
ROOFTOP GREENING GUIDE - PLANT SELECTION

Suitable Plants for Melbourne Rooftop Gardens:

SEDUMS AND SUCCULENTS:
- Sedum spurium: Drought-resistant, handles full sun, weight: very light, ideal for extensive green roofs
- Sedum acre: Low maintenance, spreads easily, excellent for flat rooftops
- Echeveria: Handles dry conditions, minimal soil depth required (5-10cm)

NATIVE AUSTRALIAN PLANTS:
- Lomandra longifolia: Hardy grass, drought tolerant, suits Melbourne climate
- Dianella revoluta: Low water needs, handles wind exposure well
- Scaevola aemula: Compact, handles heat and wind, good for exposed rooftops

HERBS AND EDIBLES:
- Thyme: Drought tolerant, aromatic, suits shallow soil (10-15cm)
- Rosemary: Handles full sun and wind, minimal maintenance
- Mint: Needs more water, suitable for irrigated rooftop beds

PLANTS TO AVOID:
- Large trees: Too heavy, root damage risk
- Plants needing deep soil (>30cm) for extensive roofs
- High water demand plants without irrigation system
"""

doc2 = """
ROOFTOP GREENING GUIDE - INSTALLATION AND COSTS

GREEN ROOF TYPES:

EXTENSIVE GREEN ROOF:
- Soil depth: 5-15cm
- Weight load: 60-150 kg/m²
- Cost: AUD $150-300 per m²
- Maintenance: Low (1-2 times per year)
- Best for: Sedums, grasses, herbs
- Suitable for most residential rooftops

INTENSIVE GREEN ROOF:
- Soil depth: 15-100cm
- Weight load: 180-500 kg/m²
- Cost: AUD $300-800 per m²
- Maintenance: High (regular watering, pruning)
- Best for: Shrubs, vegetables, small trees
- Requires structural engineer assessment

SEMI-INTENSIVE GREEN ROOF:
- Soil depth: 10-25cm
- Weight load: 120-200 kg/m²
- Cost: AUD $200-400 per m²
- Maintenance: Moderate
- Best for: Mix of sedums and small shrubs

INSTALLATION STEPS:
1. Structural assessment by engineer (AUD $500-1500)
2. Waterproofing membrane installation
3. Root barrier layer
4. Drainage layer
5. Filter fabric
6. Growing medium/substrate
7. Plant installation
"""

doc3 = """
ROOFTOP GREENING GUIDE - PERMITS AND REGULATIONS MELBOURNE

CITY OF MELBOURNE REQUIREMENTS:

PLANNING PERMITS:
- Green roofs under 10m² generally do not require a planning permit
- Green roofs over 10m² may require a planning permit depending on zoning
- Contact Melbourne City Council on 03 9658 9658 for specific advice
- Apply through Victorian Planning Authority portal: planning.vic.gov.au

BUILDING PERMITS:
- Any structural modifications require a building permit
- Submit to your local council building department
- Require engineer certificate for loads over 150 kg/m²
- Typical processing time: 4-8 weeks

STRATA AND BODY CORPORATE:
- Apartment rooftops require body corporate approval
- Submit written proposal with structural engineer report
- Allow 4-12 weeks for approval process

HERITAGE OVERLAYS:
- Properties in heritage overlays require Heritage Victoria approval
- Additional assessments may be needed
- Contact Heritage Victoria: heritage.vic.gov.au

SUSTAINABILITY INCENTIVES:
- City of Melbourne Green Your Laneway grants: up to AUD $5000
- Victorian Government Greener Government Buildings program
- Some councils offer rate rebates for green infrastructure
"""

doc4 = """
ROOFTOP GREENING GUIDE - GREEN SCORE EXPLAINED

LEAFY HAVEN GREEN SCORE SYSTEM:

The green score rates rooftop greening potential from 0-100.

SCORE COMPONENTS:
1. Existing Vegetation Coverage (25 points)
   - 0-10%% coverage: 0-8 points
   - 10-30%% coverage: 8-16 points
   - 30%% + coverage: 16-25 points

2. Rooftop Area (25 points)
   - Under 20m²: 0-8 points
   - 20-50m²: 8-16 points
   - Over 50m²: 16-25 points

3. Sun Exposure (25 points)
   - North-facing (best in Melbourne): 20-25 points
   - East/West-facing: 12-18 points
   - South-facing (least sun): 0-10 points

4. Structural Suitability (25 points)
   - Flat roof: 20-25 points
   - Slight slope (<10 degrees): 12-18 points
   - Steep slope (>10 degrees): 0-10 points

SCORE INTERPRETATION:
- 0-25: Low potential — structural or sun exposure challenges
- 26-50: Moderate potential — extensive green roof recommended
- 51-75: Good potential — semi-intensive or extensive roof suitable
- 76-100: Excellent potential — full green roof highly recommended

IMPROVING YOUR SCORE:
- Add irrigation system to support more plant variety
- Install lightweight substrate to reduce structural load
- Consider vertical gardens if rooftop area is limited
"""

doc5 = """
ROOFTOP GREENING GUIDE - MAINTENANCE AND ENVIRONMENTAL BENEFITS

MAINTENANCE SCHEDULE:

SPRING (September-November):
- Inspect waterproofing membrane
- Fertilise with slow-release organic fertiliser
- Plant new seedlings
- Check and clear drainage points

SUMMER (December-February):
- Water 2-3 times per week without irrigation system
- Monitor for pest activity
- Trim overgrown plants
- Check for heat stress in plants

AUTUMN (March-May):
- Reduce watering frequency
- Remove dead plant material
- Inspect root barriers
- Prepare for winter with mulching

WINTER (June-August):
- Minimal maintenance needed
- Check drainage after heavy rain
- Melbourne average winter rainfall: 150mm — usually sufficient

ENVIRONMENTAL BENEFITS:
- Temperature reduction: Green roofs reduce rooftop temperature by 20-40°C
- Stormwater management: Absorbs 50-90%% of rainfall
- Urban heat island: Can reduce surrounding air temperature by 1-3°C
- Biodiversity: Provides habitat for birds and insects
- Air quality: Filters particulate matter and CO2
- Building energy: Reduces cooling costs by 15-25%%
- Noise reduction: 8-15 decibel reduction in building noise

WATER USAGE:
- Extensive roof: 2-4 litres/m²/week in summer
- Intensive roof: 8-15 litres/m²/week in summer
- Rainwater harvesting recommended to offset usage
"""

docs = {
    "plant_selection.txt": doc1,
    "installation_costs.txt": doc2,
    "permits_regulations.txt": doc3,
    "green_score_explained.txt": doc4,
    "maintenance_benefits.txt": doc5
}

for filename, content in docs.items():
    with open(f"documents/{filename}", "w") as f:
        f.write(content)

print(f"✓ Created {len(docs)} knowledge base documents")
for filename in docs.keys():
    print(f"  - {filename}")

✓ Created 5 knowledge base documents
  - plant_selection.txt
  - installation_costs.txt
  - permits_regulations.txt
  - green_score_explained.txt
  - maintenance_benefits.txt


In [6]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Load all documents
print("Loading documents...")
documents = []
for filename in os.listdir("documents"):
    if filename.endswith(".txt"):
        loader = TextLoader(f"documents/{filename}")
        documents.extend(loader.load())
        print(f"  ✓ Loaded {filename}")

# Split into chunks
print("\nChunking documents...")
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"  ✓ Created {len(chunks)} chunks from {len(documents)} documents")

# Load embedding model
print("\nLoading embedding model (first time may take 1-2 mins)...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("  ✓ Embedding model loaded")

# Store in ChromaDB
print("\nStoring in ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)
print(f"  ✓ Stored {len(chunks)} chunks in ChromaDB")
print("\n✓ Knowledge base ready!")

Loading documents...
  ✓ Loaded green_score_explained.txt
  ✓ Loaded installation_costs.txt
  ✓ Loaded plant_selection.txt
  ✓ Loaded maintenance_benefits.txt
  ✓ Loaded permits_regulations.txt

Chunking documents...
  ✓ Created 16 chunks from 5 documents

Loading embedding model (first time may take 1-2 mins)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  ✓ Embedding model loaded

Storing in ChromaDB...


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


  ✓ Stored 16 chunks in ChromaDB

✓ Knowledge base ready!


In [7]:
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Connect to existing vectorstore
vectorstore = Chroma(
    persist_directory="./chroma_db",
    embedding_function=embeddings
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

# Custom prompt template
prompt_template = """You are a helpful assistant for Leafy Haven, an AI rooftop greening application for Melbourne, Australia.
Use the following information to answer the user's question accurately and helpfully.
If the answer is not in the provided information, say "I don't have specific information about that, please contact the City of Melbourne council for guidance."

Context:
{context}

Question: {question}

Helpful Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Build RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("✓ RAG chain ready!")

# Quick test
print("\nTesting RAG chain...")
result = rag_chain.invoke({"query": "What does a green score of 65 mean for my rooftop?"})
print("\nAnswer:", result["result"])

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


✓ RAG chain ready!

Testing RAG chain...

Answer: A green score of 65 falls within the range of 51-75, which indicates "Good potential" for rooftop greening. This means that a semi-intensive or extensive roof is suitable for your rooftop.


In [12]:
import shutil
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_groq import ChatGroq

# Recreate all variables
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

prompt_template = """You are a helpful assistant for Leafy Haven, an AI rooftop greening application for Melbourne, Australia.
Use the following information to answer the user's question accurately and helpfully.
If the answer is not in the provided information, say "I don't have specific information about that, please contact the City of Melbourne council for guidance."

Context:
{context}

Question: {question}

Helpful Answer:"""

prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

# Save new documents
doc6 = """
LEAFY HAVEN - UNDERSTANDING YOUR VISUALISATION RESULTS

WHAT THE STABLE DIFFUSION VISUALISATION SHOWS:
The green visualisation image shows a realistic preview of what your rooftop could look like after greening.
This is an AI-generated image using Stable Diffusion inpainting technology.

HOW TO INTERPRET YOUR VISUALISATION:
- Green overlay areas: Zones identified as suitable for plant installation
- Darker green areas: Higher density planting recommended
- Lighter green areas: Low maintenance ground cover recommended
- No overlay areas: Structurally unsuitable or already vegetated zones

IMPORTANT NOTES ABOUT THE VISUALISATION:
- The image is an AI preview only, not an architectural plan
- Actual plant species and layout should be confirmed with a landscape architect
- The visualisation assumes standard soil depth of 10cm
- Colours in the visualisation represent vegetation density, not specific plant species

WHAT TO DO AFTER SEEING YOUR VISUALISATION:
1. Note your green score and save your visualisation image
2. Share with a landscape architect or green roof installer for professional advice
3. Contact your local council to check permit requirements
4. Get 2-3 quotes from certified green roof installers
5. Consider joining the City of Melbourne Urban Greening program
"""

doc7 = """
LEAFY HAVEN - NEXT STEPS AFTER YOUR ROOFTOP ANALYSIS

SCORE 0-25 (LOW POTENTIAL):
- Your rooftop may have structural or sun exposure challenges
- Consider vertical gardens on walls or balconies instead
- Explore indoor plant walls as an alternative
- Contact a structural engineer to assess improvement options
- Estimated cost to improve suitability: AUD $2000-5000

SCORE 26-50 (MODERATE POTENTIAL):
- Extensive green roof is your best option
- Start small with a test area of 5-10m²
- Recommended first plants: Sedums and native grasses
- Get a structural assessment before proceeding
- Expected installation timeline: 4-8 weeks after permits
- Connect with: City of Melbourne Green Roof Network

SCORE 51-75 (GOOD POTENTIAL):
- Semi-intensive or extensive roof both viable
- Consider a mix of ornamental and edible plants
- Irrigation system recommended for best results
- Apply for City of Melbourne greening grants
- Engage a landscape architect for design
- Expected installation timeline: 6-10 weeks after permits

SCORE 76-100 (EXCELLENT POTENTIAL):
- Full green roof highly recommended
- Both intensive and extensive options available
- Opportunity to create a rooftop garden or urban farm
- High value addition to property
- Eligible for maximum government incentives
- Consider rooftop beekeeping or composting additions
- Expected installation timeline: 8-16 weeks after permits

FINDING PROFESSIONAL HELP IN MELBOURNE:
- Green Roofs Australasia: greenroofs.org.au
- Landscape Architecture Australia: landscapeaustralia.com.au
- City of Melbourne Urban Forest: melbourne.vic.gov.au/urbanforest
"""

doc8 = """
LEAFY HAVEN - FREQUENTLY ASKED QUESTIONS

Q: How accurate is the green score?
A: The green score is based on computer vision analysis of your uploaded image. It analyses vegetation coverage, rooftop area, sun exposure and structural suitability. Accuracy is highest with clear, high resolution aerial or rooftop images taken in daylight. The score is a guide only and should be validated by a professional assessor.

Q: What type of image should I upload to Leafy Haven?
A: For best results upload a clear aerial or top-down photograph of your rooftop taken in daylight. Avoid images with heavy shadows, obstructions or low resolution. Google Maps satellite view screenshots also work well.

Q: Can I use Leafy Haven for any rooftop in Australia?
A: Leafy Haven is optimised for Melbourne rooftops and uses Melbourne-specific climate data, council regulations and plant databases. Results for rooftops outside Melbourne may be less accurate for permit and plant recommendations.

Q: Is my rooftop image stored or shared?
A: Images uploaded to Leafy Haven are processed in real time and are not stored permanently or shared with third parties.

Q: Can Leafy Haven analyse multiple rooftops at once?
A: Currently Leafy Haven analyses one rooftop per submission. For multiple rooftops submit each image separately.

Q: What if my rooftop has existing vegetation?
A: Leafy Haven detects existing vegetation using computer vision and factors this into your green score. Existing vegetation positively contributes to your score.

Q: How often should I re-analyse my rooftop?
A: Re-analyse after any major changes to the rooftop structure, after installing new vegetation, or annually to track your greening progress over time.

Q: What is the difference between Leafy Haven and a landscape architect?
A: Leafy Haven provides an AI-powered preliminary assessment to help you understand your rooftop's potential quickly and for free. A landscape architect provides detailed professional design, engineering sign-off and project management for actual installation.
"""

new_docs = {
    "visualisation_guide.txt": doc6,
    "next_steps.txt": doc7,
    "faq.txt": doc8
}

os.makedirs("documents", exist_ok=True)
for filename, content in new_docs.items():
    with open(f"documents/{filename}", "w") as f:
        f.write(content)
    print(f"  ✓ Saved {filename}")

# Use fresh ChromaDB folder
import time
chroma_dir = f"./chroma_db_{int(time.time())}"
print(f"✓ Using fresh ChromaDB folder: {chroma_dir}")

# Load all documents
print("\nRebuilding knowledge base...")
documents = []
for filename in os.listdir("documents"):
    if filename.endswith(".txt"):
        loader = TextLoader(f"documents/{filename}")
        documents.extend(loader.load())
        print(f"  ✓ Loaded {filename}")

chunks = splitter.split_documents(documents)
print(f"✓ Total chunks: {len(chunks)}")

# Rebuild vectorstore
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=chroma_dir
)

# Rebuild RAG chain
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt},
    return_source_documents=True
)

print("\n✓ Knowledge base rebuilt with all 8 documents!")

# Quick test
result = rag_chain.invoke({"query": "What should I do after getting a score of 80?"})
print("\nTest answer:", result["result"])

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  ✓ Saved visualisation_guide.txt
  ✓ Saved next_steps.txt
  ✓ Saved faq.txt
✓ Using fresh ChromaDB folder: ./chroma_db_1781193122

Rebuilding knowledge base...
  ✓ Loaded green_score_explained.txt
  ✓ Loaded installation_costs.txt
  ✓ Loaded faq.txt
  ✓ Loaded plant_selection.txt
  ✓ Loaded maintenance_benefits.txt
  ✓ Loaded next_steps.txt
  ✓ Loaded permits_regulations.txt
  ✓ Loaded visualisation_guide.txt
✓ Total chunks: 31


ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



✓ Knowledge base rebuilt with all 8 documents!

Test answer: Congratulations on achieving an excellent potential score of 80. Based on the information provided, with a score of 76-100, a full green roof is highly recommended. This means you have a high potential for creating a rooftop garden or urban farm, which can add significant value to your property.

Considering your score, you may want to explore the following options:

1. **Create a rooftop garden or urban farm**: With an excellent potential score, you can consider installing a full green roof, which can provide a unique opportunity to grow a variety of plants and potentially even farm some produce.
2. **Take advantage of government incentives**: As your score indicates excellent potential, you may be eligible for maximum government incentives for rooftop greening projects.
3. **Consider additional features**: You may also want to explore adding features like rooftop beekeeping or composting to further enhance your rooftop gre

In [14]:
import gradio as gr

def ask_leafy_haven(question, history):
    if not question.strip():
        return "Please ask a question about rooftop greening!"

    result = rag_chain.invoke({"query": question})
    answer = result["result"]

    # Show which documents were used
    sources = set()
    for doc in result["source_documents"]:
        source = doc.metadata.get("source", "").replace("documents/", "").replace(".txt", "").replace("_", " ").title()
        sources.add(source)

    if sources:
        answer += f"\n\n📚 *Sources: {', '.join(sources)}*"

    return answer

# Build the interface
with gr.Blocks(theme=gr.themes.Soft(), title="Leafy Haven Assistant") as app:
    gr.Markdown("""
    # 🌿 Leafy Haven AI Assistant
    ### Your rooftop greening guide for Melbourne
    Ask me anything about green scores, plant selection, installation costs, permits, maintenance, visualisation results, or next steps after your analysis!
    """)

    chatbot = gr.Chatbot(height=400, placeholder="Ask me about your rooftop greening project...")

    with gr.Row():
        msg = gr.Textbox(
            placeholder="e.g. What plants suit a north-facing rooftop?",
            scale=4,
            show_label=False
        )
        submit = gr.Button("Ask 🌱", scale=1, variant="primary")

    gr.Examples(
        examples=[
            "What does a green score of 45 mean?",
            "What plants work best for Melbourne rooftops?",
            "How much does a green roof cost per square metre?",
            "Do I need a permit for rooftop greening in Melbourne?",
            "How do I maintain a green roof in summer?",
            "What are the environmental benefits of rooftop greening?",
            "What does the green overlay in my visualisation mean?",
            "I got a score of 80, what should I do next?",
            "Is my uploaded image stored by Leafy Haven?",
            "How is Leafy Haven different from a landscape architect?"
        ],
        inputs=msg
    )

    def respond(message, chat_history):
        answer = ask_leafy_haven(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history

    submit.click(respond, [msg, chatbot], [msg, chatbot])
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

app.launch(share=True)

/tmp/ipykernel_8809/135850202.py:22: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="Leafy Haven Assistant") as app:
/tmp/ipykernel_8809/135850202.py:29: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=400, placeholder="Ask me about your rooftop greening project...")
/tmp/ipykernel_8809/135850202.py:29: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=400, plac

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5bc63e97015a6e551.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
